<a href="https://colab.research.google.com/github/Elamathi995/Variant-calling/blob/main/Germline_variant_calling_using_GATK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


SRA toolkit installation

In [ ]:
!wget https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz

--2026-04-26 06:02:58--  https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz
Resolving ftp-trace.ncbi.nlm.nih.gov (ftp-trace.ncbi.nlm.nih.gov)... 130.14.250.31, 130.14.250.7, 2607:f220:41e:250::12, ...
Connecting to ftp-trace.ncbi.nlm.nih.gov (ftp-trace.ncbi.nlm.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 89143120 (85M) [application/x-gzip]
Saving to: ‘sratoolkit.current-ubuntu64.tar.gz’

sratoolkit.current- 100%[===================>]  85.01M  48.0MB/s    in 1.8s    

2026-04-26 06:03:00 (48.0 MB/s) - ‘sratoolkit.current-ubuntu64.tar.gz’ saved [89143120/89143120]



Unzipping the SRA Toolkit

if you want to see how each step is happening, use the -v (verbose) command.

In [ ]:
!tar -xzf sratoolkit.current-ubuntu64.tar.gz

In [ ]:
!ls

drive	     sratoolkit.3.4.1-ubuntu64
sample_data  sratoolkit.current-ubuntu64.tar.gz


for clarity renaming the sratoolkit with version name

In [ ]:
!mv sratoolkit.3.4.1-ubuntu64 sratoolkit

Adding to the path

In [ ]:
import os
os.environ["PATH"] += ":/content/sratoolkit/bin"

Checking the installations, you can skip prefetch because fasterq-dump can download and convert .sra files directly to fastq, but using prefetch is cleaner and prevents errors.

In [ ]:
!which prefetch
!which fasterq-dump

/content/sratoolkit/bin/prefetch
/content/sratoolkit/bin/fasterq-dump


In [ ]:
!fasterq-dump --help


Usage:
  fasterq-dump <path> [options]
  fasterq-dump <accession> [options]

Options:
  -F|--format                      format (special, fastq, default=fastq) 
  -o|--outfile                     output-file 
  -O|--outdir                      output-dir 
  -b|--bufsize                     size of file-buffer dflt=1MB 
  -c|--curcache                    size of cursor-cache dflt=10MB 
  -m|--mem                         memory limit for sorting dflt=100MB 
  -t|--temp                        where to put temp. files dflt=curr dir 
  -e|--threads                     how many thread dflt=6 
  -p|--progress                    show progress 
  -x|--details                     print details 
  -s|--split-spot                  split spots into reads 
  -S|--split-files                 write reads into different files 
  -3|--split-3                     writes single reads in special file 
  --concatenate-reads              writes whole spots into one file 
  -Z|--stdout                      p

In [ ]:
!prefetch --help

Usage:
  prefetch [options] <SRA accession> [...]
  Download SRA files and their dependencies

  prefetch [options] --perm <JWT cart file> <SRA accession> [...]
  Download SRA files and their dependencies from JWT cart

  prefetch [options] --cart <kart file>
  Download cart file

  prefetch [options] <URL> --output-file <FILE>
  Download URL to FILE

  prefetch [options] <URL> [...] --output-directory <DIRECTORY>
  Download URL or URL-s to DIRECTORY

  prefetch [options] <SRA file> [...]
  Check SRA file for missed dependencies and download them


Options:
  -T|--type <value>                Specify file type to download. Default: sra 
  -t|--transport <http|fasp|both>  Transport: one of: fasp; http; both 
                                   [default]. (fasp only; http only; first try 
                                   fasp (ascp), use http if cannot download 
                                   using fasp). 
  --location <value>               Location of data. 

  -N|--min-size <size> 

This version checking is optional, as I have renamed the sratoolkit for the pipeline

In [ ]:
!prefetch --version
!fasterq-dump --version

prefetch : 3.4.1

fasterq-dump : 3.4.1



All installations are working fine

Now I will remove the compressed sratoolkit file to save disk space

In [ ]:
!rm sratoolkit.current-ubuntu64.tar.gz

Starting the pipeline by downloading the sequence data

For reusability, and building a cleaner plus scalable pipeline, I have created the variable SRR_ID. This step can be skipped and preftech can directly be used with the actual SRR ID. This additional command --max-size 100G is used to tell SRA toolkit to allow maximum size of downloadable file is to be 100GB. This is important as SRA toolkit has a default limit of 20GB download limit.a

In [ ]:
SRR_ID="SRR25243226"
!prefetch --max-size 100G $SRR_ID

2026-04-26T06:03:45 prefetch.3.4.1: 1) Resolving 'SRR25243226'...
2026-04-26T06:03:46 prefetch.3.4.1: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-04-26T06:03:46 prefetch.3.4.1: 1) Downloading 'SRR25243226'...
2026-04-26T06:03:46 prefetch.3.4.1:  SRA Normalized Format file is being retrieved
2026-04-26T06:03:46 prefetch.3.4.1:  Downloading via HTTPS...
2026-04-26T06:06:41 prefetch.3.4.1:  HTTPS download succeed
2026-04-26T06:06:53 prefetch.3.4.1:  'SRR25243226' is valid: 4281405130 bytes were streamed from 4281405063
2026-04-26T06:06:53 prefetch.3.4.1: 1) 'SRR25243226' was downloaded successfully
2026-04-26T06:06:53 prefetch.3.4.1: 1) Resolving 'SRR25243226's dependencies...
2026-04-26T06:06:54 prefetch.3.4.1: 'SRR25243226' has 0 unresolved dependencies


checking the prefetch results

In [ ]:
!find / -name "SRR25243226.sra" 2>/dev/null

/content/SRR25243226/SRR25243226.sra


So, my file exists in the content/SRR25243226 folder.

Converting SRR25243226.sra file into FASTQ format

In [ ]:
!fasterq-dump $SRR_ID --split-files --threads 4 --temp /content/temporary

spots read      : 40,401,208
reads read      : 80,802,416
reads written   : 80,802,416


I created a temporary folder for storing the intermediate files

In [ ]:
!ls -lh

total 34G
drwx------ 5 root root 4.0K Apr 26 06:02 drive
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data
drwxrwxr-x 5 root root 4.0K Mar 25 16:47 sratoolkit
drwxr-xr-x 2 root root 4.0K Apr 26 06:06 SRR25243226
-rw-r--r-- 1 root root  17G Apr 26 06:35 SRR25243226_1.fastq
-rw-r--r-- 1 root root  17G Apr 26 06:35 SRR25243226_2.fastq
drwxr-xr-x 2 root root 4.0K Apr 26 06:35 temporary


In [ ]:
!du -h SRR25243226_*.fastq

17G	SRR25243226_1.fastq
17G	SRR25243226_2.fastq


In [ ]:
!head SRR25243226_1.fastq

@SRR25243226.1 A00192:269:HLLY2DSXX:1:1101:3233:1016 length=150
TNTTACGAACACTATCCAACAATTAATGATGATTTTAAATTCAGCAAGTGATCAACCTTCAGAAAATCTGATTTCCTATTTTAACGTAAGCCATATATGAAACATTATTTATTGTAATATCTTGGCAAAGAAACTTGAGATCGGAAGAGC
+SRR25243226.1 A00192:269:HLLY2DSXX:1:1101:3233:1016 length=150
F#FFF,:FFFFFF:FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF::FFFFFFFFFFFFFFFFFFFFFFFFFF:FFFFFFFF:,FFFFFFFFFFFFFFFFF:FFFFF:FFFFFFFFFFF:FFFFFFFFFFFFF:FFFFFF:FFF::FF,
@SRR25243226.2 A00192:269:HLLY2DSXX:1:1101:4806:1016 length=150
TNGATGTGTCTGTTCCCAAGGTAGAAGGTGAAATGAAAGTGCCAGATGTTGACATCAGAGGGCCCAAAGTGGACATTGATGCCCCAGATGTGGATGTTCATGGCCCAGACTGGCACCTGAAGATGCCTAAGATGAAAATGCCCAAGTTCA
+SRR25243226.2 A00192:269:HLLY2DSXX:1:1101:4806:1016 length=150
F#FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF,FFFFFFFFFFFFFFFFFFFFFFFFF:FFFFFFFF:FFFFFFFFFFFFFFFFFFFFFFFFFFFFFF:FFFFFFFFFFFFFFF:FFFFFFFFFFFFFFFFFFFFF
@SRR25243226.3 A00192:269:HLLY2DSXX:1:1101:5312:1016 length=150
ANTCCTCGAATCGAGGAGCAAGAACTTTCTGAAAATACAAGCCTTCCTGCAGAAGAAGCAAACGGGAGCCTTTCTG

starting with the QC process by updating packages and installing fastQC

In [ ]:
!apt-get update -y > /dev/null 2>&1
!apt-get install -y fastqc > /dev/null 2>&1

checking whether fastQC has installed or not

In [ ]:
!fastqc --version

FastQC v0.11.9


Doing quality check with fastQC

In [ ]:
!fastqc SRR25243226_1.fastq SRR25243226_2.fastq

Started analysis of SRR25243226_1.fastq
Approx 5% complete for SRR25243226_1.fastq
Approx 10% complete for SRR25243226_1.fastq
Approx 15% complete for SRR25243226_1.fastq
Approx 20% complete for SRR25243226_1.fastq
Approx 25% complete for SRR25243226_1.fastq
Approx 30% complete for SRR25243226_1.fastq
Approx 35% complete for SRR25243226_1.fastq
Approx 40% complete for SRR25243226_1.fastq
Approx 45% complete for SRR25243226_1.fastq
Approx 50% complete for SRR25243226_1.fastq
Approx 55% complete for SRR25243226_1.fastq
Approx 60% complete for SRR25243226_1.fastq
Approx 65% complete for SRR25243226_1.fastq
Approx 70% complete for SRR25243226_1.fastq
Approx 75% complete for SRR25243226_1.fastq
Approx 80% complete for SRR25243226_1.fastq
Approx 85% complete for SRR25243226_1.fastq
Approx 90% complete for SRR25243226_1.fastq
Approx 95% complete for SRR25243226_1.fastq
Analysis complete for SRR25243226_1.fastq
Started analysis of SRR25243226_2.fastq
Approx 5% complete for SRR25243226_2.fastq


After generating FastQC report, I found Illumina universal adapter and low quality base reads. So, I will do the trimming using Trimmomatic.

Trimmomatic installation for trimming

In [ ]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


since java is already installed, no need to reinstall. I ll proceed with trimmomatic directly.

In [ ]:
!apt-get update

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:14 http://

In [ ]:
!apt-get install -y trimmomatic > /dev/null 2>&1

In [ ]:
!trimmomatic -version

/bin/bash: line 1: trimmomatic: command not found


checking installed files and the version using java command

In [ ]:
!dpkg -L trimmomatic

/.
/usr
/usr/bin
/usr/bin/TrimmomaticPE
/usr/share
/usr/share/doc
/usr/share/doc/trimmomatic
/usr/share/doc/trimmomatic/README.Debian
/usr/share/doc/trimmomatic/README.test
/usr/share/doc/trimmomatic/changelog.Debian.gz
/usr/share/doc/trimmomatic/copyright
/usr/share/doc/trimmomatic/examples
/usr/share/doc/trimmomatic/examples/batchTrim
/usr/share/doc/trimmomatic/run-unit-test
/usr/share/doc/trimmomatic/test_data
/usr/share/doc/trimmomatic/test_data/test_forward.fq.gz
/usr/share/doc/trimmomatic/test_data/test_reverse.fq.gz
/usr/share/doc/trimmomatic/test_data/test_single.fq.gz
/usr/share/java
/usr/share/java/trimmomatic-0.39.jar
/usr/share/man
/usr/share/man/man1
/usr/share/man/man1/TrimmomaticPE.1.gz
/usr/share/trimmomatic
/usr/share/trimmomatic/NexteraPE-PE.fa
/usr/share/trimmomatic/TruSeq2-PE.fa
/usr/share/trimmomatic/TruSeq2-SE.fa
/usr/share/trimmomatic/TruSeq3-PE-2.fa
/usr/share/trimmomatic/TruSeq3-PE.fa
/usr/share/trimmomatic/TruSeq3-SE.fa
/usr/bin/TrimmomaticSE
/usr/share/java/t

In [ ]:
!TrimmomaticPE -version

0.39


Running trimmomatic

In [ ]:
!TrimmomaticPE -threads 4 \
SRR25243226_1.fastq SRR25243226_2.fastq \
SRR_25243226_trimmed_1_paired.fastq SRR_25243226_trimmed_1_unpaired.fastq \
SRR_25243226_trimmed_2_paired.fastq SRR_25243226_trimmed_2_unpaired.fastq \
ILLUMINACLIP:/usr/share/trimmomatic/TruSeq3-PE.fa:2:30:10 \
SLIDINGWINDOW:4:20 \
MINLEN:50

TrimmomaticPE: Started with arguments:
 -threads 4 SRR25243226_1.fastq SRR25243226_2.fastq SRR_25243226_trimmed_1_paired.fastq SRR_25243226_trimmed_1_unpaired.fastq SRR_25243226_trimmed_2_paired.fastq SRR_25243226_trimmed_2_unpaired.fastq ILLUMINACLIP:/usr/share/trimmomatic/TruSeq3-PE.fa:2:30:10 SLIDINGWINDOW:4:20 MINLEN:50
Using PrefixPair: 'TACACTCTTTCCCTACACGACGCTCTTCCGATCT' and 'GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCT'
ILLUMINACLIP: Using 1 prefix pairs, 0 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences
Quality encoding detected as phred33
Input Read Pairs: 40401208 Both Surviving: 31106801 (76.99%) Forward Only Surviving: 7083657 (17.53%) Reverse Only Surviving: 1002078 (2.48%) Dropped: 1208672 (2.99%)
TrimmomaticPE: Completed successfully


I have to save the trimmed files for future sessions so that I won't have to rerun the entire pipeline. But to save drive space, I am compressing trimmed files. FastQC can directly use compressed files. So its efficient.

checking results of trimmomatic

In [ ]:
!ls -lh SRR_25243226_trimmed_*
!ls -lh /content/drive/MyDrive/ | grep SRR_trimmed

-rw-r--r-- 1 root root  13G Apr 26 06:50 SRR_25243226_trimmed_1_paired.fastq
-rw-r--r-- 1 root root 2.6G Apr 26 06:50 SRR_25243226_trimmed_1_unpaired.fastq
-rw-r--r-- 1 root root  13G Apr 26 06:50 SRR_25243226_trimmed_2_paired.fastq
-rw-r--r-- 1 root root 375M Apr 26 06:50 SRR_25243226_trimmed_2_unpaired.fastq


In [ ]:
# remove unpaired
!rm SRR_25243226_trimmed_*unpaired.fastq

As the disk space is getting full, I am removing the unwanted files.

In [ ]:
!rm -r /content/SRR25243226

You can also compress the trimmed reads and save it

removing raw fastq files

In [ ]:
!rm SRR25243226_1.fastq
!rm SRR25243226_2.fastq

Performing fastqc check on the trimmed files

In [ ]:
!fastqc SRR_25243226_trimmed_1_paired.fastq SRR_25243226_trimmed_2_paired.fastq

Started analysis of SRR_25243226_trimmed_1_paired.fastq
Approx 5% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 10% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 15% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 20% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 25% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 30% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 35% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 40% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 45% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 50% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 55% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 60% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 65% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 70% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 75% complete for SRR_25243226_trimmed_1_paired.fastq
Approx 80% complete for SRR_25243226_trimmed_

checking the results of 2nd fastqc after trimming

In [ ]:
!ls -1 *_fastqc.html

In [ ]:
!ls -1 *_fastqc.*

downloading trimmed results

In [ ]:
from google.colab import files
files.download('SRR_25243226_trimmed_1_paired_fastqc.html')
files.download('SRR_25243226_trimmed_2_paired_fastqc.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Since trimming and quality control are perfect, I am going for alignment now.

checking everything is fine before alignment

In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   47G   62G  44% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G     0  5.8G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1       114G   91G   24G  80% /kaggle/input
tmpfs           6.4G   92K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive            15G   14G  1.6G  90% /content/drive


In [ ]:
!ls -lh SRR_25243226_trimmed_*paired.fastq

-rw-r--r-- 1 root root 13G Apr 26 06:50 SRR_25243226_trimmed_1_paired.fastq
-rw-r--r-- 1 root root 13G Apr 26 06:50 SRR_25243226_trimmed_2_paired.fastq


installing bwa for alignment

In [ ]:
!apt-get install -y bwa samtools > /dev/null 2>&1

downloading reference genome

In [ ]:
!wget https://ftp.ebi.ac.uk/pub/databases/ena/reference/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

--2026-04-23 05:54:58--  https://ftp.ebi.ac.uk/pub/databases/ena/reference/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-23 05:54:58 ERROR 404: Not Found.



i used ena url which is wrong. i have to use ncbi ensembl

In [ ]:
!wget https://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

--2026-04-27 15:01:14--  https://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 881964081 (841M) [application/x-gzip]
Saving to: ‘Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz’

Homo_sapiens.GRCh38 100%[===================>] 841.11M  38.3MB/s    in 22s     

2026-04-27 15:01:37 (38.3 MB/s) - ‘Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz’ saved [881964081/881964081]



unzipping the reference genome

In [ ]:
!gunzip Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

verifying unzipping process

In [ ]:
!ls -lh Homo_sapiens.GRCh38.dna.primary_assembly.fa

-rw-r--r-- 1 root root 3.0G Apr 21  2023 Homo_sapiens.GRCh38.dna.primary_assembly.fa


indexing reference genome

In [ ]:
!bwa index Homo_sapiens.GRCh38.dna.primary_assembly.fa

[bwa_index] Pack FASTA... ^C


checking the results of indexing

In [ ]:
!ls -lh Homo_sapiens.GRCh38.dna.primary_assembly.fa.*

-rw-r--r-- 1 root root  18K Apr 23 06:59 Homo_sapiens.GRCh38.dna.primary_assembly.fa.amb
-rw-r--r-- 1 root root  17K Apr 23 06:59 Homo_sapiens.GRCh38.dna.primary_assembly.fa.ann
-rw-r--r-- 1 root root 2.9G Apr 23 06:58 Homo_sapiens.GRCh38.dna.primary_assembly.fa.bwt
-rw-r--r-- 1 root root 740M Apr 23 06:59 Homo_sapiens.GRCh38.dna.primary_assembly.fa.pac
-rw-r--r-- 1 root root 1.5G Apr 23 07:41 Homo_sapiens.GRCh38.dna.primary_assembly.fa.sa


In [ ]:
!ls -lh SRR_25243226_trimmed_*paired.fastq

-rw-r--r-- 1 root root 13G Apr 23 05:51 SRR_25243226_trimmed_1_paired.fastq
-rw-r--r-- 1 root root 13G Apr 23 05:51 SRR_25243226_trimmed_2_paired.fastq


saving results temporarily to prevent redoing indexing

In [ ]:
!cp Homo_sapiens.GRCh38.dna.primary_assembly.fa* /content/drive/MyDrive/

copying back index files from drive to colab

In [ ]:
!cp /content/drive/MyDrive/Homo_sapiens.GRCh38.dna.primary_assembly.fa* .

In [ ]:
!ls -lh /content/drive/MyDrive/Homo_sapiens.GRCh38.dna.primary_assembly.fa*

In [ ]:
!ls Homo_sapiens.GRCh38.dna.primary_assembly.fa*

Homo_sapiens.GRCh38.dna.primary_assembly.fa.amb


Doing the final alignment step now

In [ ]:
!bwa mem -t 4 Homo_sapiens.GRCh38.dna.primary_assembly.fa \
SRR_25243226_trimmed_1_paired.fastq \
SRR_25243226_trimmed_2_paired.fastq | \
samtools view -@ 4 -bS | \
samtools sort -@ 4 -o aligned_sorted.bam

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 273434 sequences (40000030 bp)...
[M::process] read 273728 sequences (40000000 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (1, 114752, 3, 0)
[M::mem_pestat] skip orientation FF as there are not enough pairs
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (202, 277, 386)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (1, 754)
[M::mem_pestat] mean and std.dev: (306.28, 130.29)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 938)
[M::mem_pestat] skip orientation RF as there are not enough pairs
[M::mem_pestat] skip orientation RR as there are not enough pairs
[M::mem_process_seqs] Processed 273434 reads in 106.153 CPU sec, 68.695 real sec
[M::process] read 274226 sequences (40000247 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (0, 114789, 0, 0)
[M::mem_pestat] skip orient

checking alignment results

In [ ]:
!ls -lh aligned_sorted.bam

-rw-r--r-- 1 root root 2.8G Apr 26 11:33 aligned_sorted.bam


checking BAM file is ok or not

In [ ]:
!samtools quickcheck aligned_sorted.bam && echo "BAM OK"

BAM OK


statistics of BAM file

In [ ]:
!samtools flagstat aligned_sorted.bam

62307349 + 0 in total (QC-passed reads + QC-failed reads)
62213602 + 0 primary
0 + 0 secondary
93747 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
62218562 + 0 mapped (99.86% : N/A)
62124815 + 0 primary mapped (99.86% : N/A)
62213602 + 0 paired in sequencing
31106801 + 0 read1
31106801 + 0 read2
61626402 + 0 properly paired (99.06% : N/A)
62116184 + 0 with itself and mate mapped
8631 + 0 singletons (0.01% : N/A)
385264 + 0 with mate mapped to a different chr
343858 + 0 with mate mapped to a different chr (mapQ>=5)


copying the aligned_sorted. bam file to drive

In [ ]:
!cp aligned_sorted.bam /content/drive/MyDrive/

checking first few entries of BAM file

In [ ]:
!samtools idxstats aligned_sorted.bam | head

1	248956422	6321325	773
10	133797422	2564325	414
11	135086622	3408515	432
12	133275309	3060603	412
13	114364328	1133269	230
14	107043718	1875441	252
15	101991189	2304187	299
16	90338345	2873330	326
17	83257441	3624882	380
18	80373285	1012067	158


indexing the bam file

In [ ]:
!samtools index aligned_sorted.bam

Dowloading bam files

In [ ]:
from google.colab import files
files.download("aligned_sorted.bam")
files.download("aligned_sorted.bam.bai")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

copying indexed BAM file to drive

In [ ]:
!cp aligned_sorted.bam.bai /content/drive/MyDrive/

In [ ]:
!ls -lh /content/drive/MyDrive/aligned_sorted*

-rw------- 1 root root 2.8G Apr 26 12:22 /content/drive/MyDrive/aligned_sorted.bam
-rw------- 1 root root 4.1M Apr 26 12:30 /content/drive/MyDrive/aligned_sorted.bam.bai


Mark duplicates

installing Picard tools

In [ ]:
!apt-get install -y picard-tools > /dev/null 2>&1

beginning mark duplicates

In [ ]:
!picard MarkDuplicates \
I=aligned_sorted.bam \
O=dedup.bam \
M=dedup_metrics.txt \
REMOVE_DUPLICATES=false

/bin/bash: line 1: picard: command not found


Since Picard is a Java-based tool, it should be run using Java, not directly. But just to ensure installation is perfect,I am checking everything.

In [ ]:
!ls /usr/share/java | grep picard

picard-2.26.10.jar
picard.jar


In [ ]:
!find / -name "picard*.jar" 2>/dev/null

/usr/share/java/picard-2.26.10.jar
/usr/share/java/picard.jar


starting mark duplicates again

In [ ]:
!java -jar /usr/share/java/picard.jar MarkDuplicates \
I=aligned_sorted.bam \
O=dedup.bam \
M=dedup_metrics.txt \
REMOVE_DUPLICATES=false

INFO	2026-04-26 12:45:57	MarkDuplicates	

********** NOTE: Picard's command line syntax is changing.
**********
********** For more information, please see:
********** https://github.com/broadinstitute/picard/wiki/Command-Line-Syntax-Transition-For-Users-(Pre-Transition)
**********
********** The command line looks like this in the new syntax:
**********
**********    MarkDuplicates -I aligned_sorted.bam -O dedup.bam -M dedup_metrics.txt -REMOVE_DUPLICATES false
**********


12:45:58.717 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/usr/share/java/gkl.jar!/com/intel/gkl/native/libgkl_compression.so
[Sun Apr 26 12:45:58 UTC 2026] MarkDuplicates INPUT=[aligned_sorted.bam] OUTPUT=dedup.bam METRICS_FILE=dedup_metrics.txt REMOVE_DUPLICATES=false    MAX_SEQUENCES_FOR_DISK_READ_ENDS_MAP=50000 MAX_FILE_HANDLES_FOR_READ_ENDS_MAP=8000 SORTING_COLLECTION_SIZE_RATIO=0.25 TAG_DUPLICATE_SET_MEMBERS=false REMOVE_SEQUENCING_DUPLICATES=false TAGGING_POLICY=DontTag CLEAR_DT=tr

deduplication

In [ ]:
!samtools index dedup.bam

checking dedup step is correctly done

In [ ]:
!cat dedup_metrics.txt

## htsjdk.samtools.metrics.StringHeader
# MarkDuplicates INPUT=[aligned_sorted.bam] OUTPUT=dedup.bam METRICS_FILE=dedup_metrics.txt REMOVE_DUPLICATES=false    MAX_SEQUENCES_FOR_DISK_READ_ENDS_MAP=50000 MAX_FILE_HANDLES_FOR_READ_ENDS_MAP=8000 SORTING_COLLECTION_SIZE_RATIO=0.25 TAG_DUPLICATE_SET_MEMBERS=false REMOVE_SEQUENCING_DUPLICATES=false TAGGING_POLICY=DontTag CLEAR_DT=true DUPLEX_UMI=false ADD_PG_TAG_TO_READS=true ASSUME_SORTED=false DUPLICATE_SCORING_STRATEGY=SUM_OF_BASE_QUALITIES PROGRAM_RECORD_ID=MarkDuplicates PROGRAM_GROUP_NAME=MarkDuplicates READ_NAME_REGEX=<optimized capture of last three ':' separated fields as numeric values> OPTICAL_DUPLICATE_PIXEL_DISTANCE=100 MAX_OPTICAL_DUPLICATE_SET_SIZE=300000 VERBOSITY=INFO QUIET=false VALIDATION_STRINGENCY=STRICT COMPRESSION_LEVEL=5 MAX_RECORDS_IN_RAM=500000 CREATE_INDEX=false CREATE_MD5_FILE=false GA4GH_CLIENT_SECRETS=client_secrets.json USE_JDK_DEFLATER=false USE_JDK_INFLATER=false
## htsjdk.samtools.metrics.StringHeader
# Start

checking statistics of deduplicated BAM file

In [ ]:
!samtools flagstat dedup.bam

62307349 + 0 in total (QC-passed reads + QC-failed reads)
62213602 + 0 primary
0 + 0 secondary
93747 + 0 supplementary
9554280 + 0 duplicates
9554280 + 0 primary duplicates
62218562 + 0 mapped (99.86% : N/A)
62124815 + 0 primary mapped (99.86% : N/A)
62213602 + 0 paired in sequencing
31106801 + 0 read1
31106801 + 0 read2
61626402 + 0 properly paired (99.06% : N/A)
62116184 + 0 with itself and mate mapped
8631 + 0 singletons (0.01% : N/A)
385264 + 0 with mate mapped to a different chr
343858 + 0 with mate mapped to a different chr (mapQ>=5)


creating directory to save deduplicated files in drive

In [ ]:
!mkdir -p /content/drive/MyDrive/variant_dedup

In [ ]:
!cp dedup.bam /content/drive/MyDrive/variant_dedup/
!cp dedup.bam.bai /content/drive/MyDrive/variant_dedup/
!cp dedup_metrics.txt /content/drive/MyDrive/variant_dedup/

In [ ]:
!ls -lh /content/drive/MyDrive/variant_dedup/

total 2.9G
-rw------- 1 root root 2.9G Apr 26 13:24 dedup.bam
-rw------- 1 root root 4.1M Apr 26 13:24 dedup.bam.bai
-rw------- 1 root root 3.3K Apr 26 13:24 dedup_metrics.txt


In [ ]:
!ls -lh Homo_sapiens.GRCh38.dna.primary_assembly.fa

ls: cannot access 'Homo_sapiens.GRCh38.dna.primary_assembly.fa': No such file or directory


copying back deduplicated BAM files to colab from drive

In [ ]:
!cp /content/drive/MyDrive/variant_dedup/dedup.bam .
!cp /content/drive/MyDrive/variant_dedup/dedup.bam.bai .

In [ ]:
!ls -lh dedup.bam dedup.bam.bai

-rw------- 1 root root 2.9G Apr 26 18:07 dedup.bam
-rw------- 1 root root 4.1M Apr 26 18:07 dedup.bam.bai


creating FASTA index of the reference genome

In [ ]:
!samtools faidx Homo_sapiens.GRCh38.dna.primary_assembly.fa

checking dedup bam file

In [ ]:
!samtools idxstats dedup.bam

1	248956422	6321325	773
10	133797422	2564325	414
11	135086622	3408515	432
12	133275309	3060603	412
13	114364328	1133269	230
14	107043718	1875441	252
15	101991189	2304187	299
16	90338345	2873330	326
17	83257441	3624882	380
18	80373285	1012067	158
19	58617616	3814591	387
2	242193529	4740643	717
20	64444167	1393137	169
21	46709983	643241	100
22	50818468	1278833	163
3	198295559	3429094	532
4	190214555	2557292	443
5	181538259	2728984	483
6	170805979	3004198	423
7	159345973	3150860	447
8	145138636	2008758	349
9	138394717	2703420	325
MT	16569	1133	1
X	156040895	2433905	369
Y	57227415	24312	24
KI270728.1	1872759	9061	1
KI270727.1	448248	8106	0
KI270442.1	392061	2142	1
KI270729.1	280839	1534	4
GL000225.1	211173	1562	0
KI270743.1	210658	252	0
GL000008.2	209709	1369	0
GL000009.2	201709	1266	0
KI270747.1	198735	79	0
KI270722.1	194050	882	0
GL000194.1	191469	3522	0
KI270742.1	186739	3385	0
GL000205.2	185591	1817	1
GL000195.1	182896	146	0
KI270736.1	181920	381	1
KI270733.1	179772	2274	3
GL000224.1	1

creating sequence dictionary with Picard

In [ ]:
!java -jar /usr/share/java/picard.jar CreateSequenceDictionary \
R=Homo_sapiens.GRCh38.dna.primary_assembly.fa \
O=Homo_sapiens.GRCh38.dna.primary_assembly.dict

INFO	2026-04-27 15:06:45	CreateSequenceDictionary	

********** NOTE: Picard's command line syntax is changing.
**********
********** For more information, please see:
********** https://github.com/broadinstitute/picard/wiki/Command-Line-Syntax-Transition-For-Users-(Pre-Transition)
**********
********** The command line looks like this in the new syntax:
**********
**********    CreateSequenceDictionary -R Homo_sapiens.GRCh38.dna.primary_assembly.fa -O Homo_sapiens.GRCh38.dna.primary_assembly.dict
**********


15:06:46.867 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/usr/share/java/gkl.jar!/com/intel/gkl/native/libgkl_compression.so
[Mon Apr 27 15:06:46 UTC 2026] CreateSequenceDictionary OUTPUT=Homo_sapiens.GRCh38.dna.primary_assembly.dict REFERENCE=Homo_sapiens.GRCh38.dna.primary_assembly.fa    TRUNCATE_NAMES_AT_WHITESPACE=true NUM_SEQUENCES=2147483647 VERBOSITY=INFO QUIET=false VALIDATION_STRINGENCY=STRICT COMPRESSION_LEVEL=5 MAX_RECORDS_IN_RAM=500000 CREAT

removing old indexed files

In [ ]:
!rm Homo_sapiens.GRCh38.dna.primary_assembly.fa.amb
!rm Homo_sapiens.GRCh38.dna.primary_assembly.fa.ann
!rm Homo_sapiens.GRCh38.dna.primary_assembly.fa.bwt
!rm Homo_sapiens.GRCh38.dna.primary_assembly.fa.pac
!rm Homo_sapiens.GRCh38.dna.primary_assembly.fa.sa
!rm Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz*

checking dictionary file existence

In [ ]:
!ls Homo_sapiens.GRCh38.dna.primary_assembly.*

Homo_sapiens.GRCh38.dna.primary_assembly.dict
Homo_sapiens.GRCh38.dna.primary_assembly.fa
Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai


installing GATK and unzipping it

In [ ]:
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.4.0.0/gatk-4.4.0.0.zip
!unzip -q gatk-4.4.0.0.zip

Adding GATK to the current path

In [ ]:
import os
os.environ["PATH"] += ":/content/gatk-4.4.0.0"

checking installation of GATK

In [ ]:
!gatk --version

Using GATK jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar --version
The Genome Analysis Toolkit (GATK) v4.4.0.0
HTSJDK Version: 3.0.5
Picard Version: 3.0.0


Checking everything is present or not

In [ ]:
!ls dedup.bam dedup.bam.bai Homo_sapiens.GRCh38.dna.primary_assembly.*

dedup.bam
dedup.bam.bai
Homo_sapiens.GRCh38.dna.primary_assembly.fa
Homo_sapiens.GRCh38.dna.primary_assembly.fa.amb
Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai


Variant Calling using GATKHaplotypeCaller

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar HaplotypeCaller \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-I dedup.bam \
-O raw_variants.vcf \
--native-pair-hmm-threads 4

15:35:46.252 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:35:46.392 INFO  HaplotypeCaller - ------------------------------------------------------------
15:35:46.408 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.4.0.0
15:35:46.408 INFO  HaplotypeCaller - For support and documentation go to https://software.broadinstitute.org/gatk/
15:35:46.409 INFO  HaplotypeCaller - Executing as root@567e790518d6 on Linux v6.6.113+ amd64
15:35:46.409 INFO  HaplotypeCaller - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
15:35:46.410 INFO  HaplotypeCaller - Start Date/Time: April 26, 2026 at 3:35:46 PM UTC
15:35:46.412 INFO  HaplotypeCaller - ------------------------------------------------------------
15:35:46.412 INFO  HaplotypeCaller - ------------------------------------------------------------
15:35:46.416 INFO  HaplotypeCaller - HTSJDK V

GATK Haplotype Caller failed because samples and Read Groups are not assigned

Checking whether readgroups are actually present or not

In [ ]:
!samtools view -H dedup.bam | grep '^@RG'

ReadGrouups are absent. So adding them now

In [ ]:
!java -jar /usr/share/java/picard.jar AddOrReplaceReadGroups \
I=dedup.bam \
O=dedup_rg.bam \
RGID=1 \
RGLB=lib1 \
RGPL=ILLUMINA \
RGPU=unit1 \
RGSM=sample1

INFO	2026-04-26 18:20:00	AddOrReplaceReadGroups	

********** NOTE: Picard's command line syntax is changing.
**********
********** For more information, please see:
********** https://github.com/broadinstitute/picard/wiki/Command-Line-Syntax-Transition-For-Users-(Pre-Transition)
**********
********** The command line looks like this in the new syntax:
**********
**********    AddOrReplaceReadGroups -I dedup.bam -O dedup_rg.bam -RGID 1 -RGLB lib1 -RGPL ILLUMINA -RGPU unit1 -RGSM sample1
**********


18:20:02.396 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/usr/share/java/gkl.jar!/com/intel/gkl/native/libgkl_compression.so
[Sun Apr 26 18:20:02 UTC 2026] AddOrReplaceReadGroups INPUT=dedup.bam OUTPUT=dedup_rg.bam RGID=1 RGLB=lib1 RGPL=ILLUMINA RGPU=unit1 RGSM=sample1    VERBOSITY=INFO QUIET=false VALIDATION_STRINGENCY=STRICT COMPRESSION_LEVEL=5 MAX_RECORDS_IN_RAM=500000 CREATE_INDEX=false CREATE_MD5_FILE=false GA4GH_CLIENT_SECRETS=client_secrets.json USE_JDK_DEF

indexing the new bam file

In [ ]:
!samtools index dedup_rg.bam

checking for presence of readgroups

In [ ]:
!samtools view -H dedup_rg.bam | grep '^@RG'

@RG	ID:1	LB:lib1	PL:ILLUMINA	SM:sample1	PU:unit1


In [ ]:
!ls dedup_rg.bam Homo_sapiens.GRCh38.dna.primary_assembly.*

dedup_rg.bam
Homo_sapiens.GRCh38.dna.primary_assembly.dict
Homo_sapiens.GRCh38.dna.primary_assembly.fa
Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai


variant calling with the new bam file

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar HaplotypeCaller \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-I dedup_rg.bam \
-O raw_variants.vcf \
--native-pair-hmm-threads 4

15:12:38.704 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:12:38.856 INFO  HaplotypeCaller - ------------------------------------------------------------
15:12:38.868 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.4.0.0
15:12:38.872 INFO  HaplotypeCaller - For support and documentation go to https://software.broadinstitute.org/gatk/
15:12:38.872 INFO  HaplotypeCaller - Executing as root@895d31e27f97 on Linux v6.6.113+ amd64
15:12:38.872 INFO  HaplotypeCaller - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
15:12:38.873 INFO  HaplotypeCaller - Start Date/Time: April 27, 2026 at 3:12:38 PM UTC
15:12:38.873 INFO  HaplotypeCaller - ------------------------------------------------------------
15:12:38.873 INFO  HaplotypeCaller - ------------------------------------------------------------
15:12:38.874 INFO  HaplotypeCaller - HTSJDK V

checking info about the raw vcf file generated

In [ ]:
!ls -lh raw_variants.vcf

-rw-r--r-- 1 root root 46M Apr 27 18:44 raw_variants.vcf


In [ ]:
!head raw_variants.vcf

##fileformat=VCFv4.2
##FILTER=<ID=LowQual,Description="Low quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=PL,Number=G,Type=Integer,Description="Normalized, Phred-scaled likelihoods for genotypes as defined in the VCF specification">
##GATKCommandLine=<ID=HaplotypeCaller,CommandLine="HaplotypeCaller --native-pair-hmm-threads 4 --output raw_variants.vcf --input dedup_rg.bam --reference Homo_sapiens.GRCh38.dna.primary_assembly.fa --use-posteriors-to-calculate-qual false --dont-use-dragstr-priors false --use-new-qual-calculator true --annotate-with-num-discovered-alleles false --heterozygosity 0.001 --indel-heterozygosity 1.25E-4 --heterozygosity

downloading raw variant.vcf file

In [ ]:
from google.colab import files
files.download('raw_variants.vcf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

installing bcf tools

In [ ]:
!apt-get update -qq
!apt-get install -y bcftools

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  python3-numpy python3-matplotlib texlive-latex-recommended
The following NEW packages will be installed:
  bcftools
0 upgraded, 1 newly installed, 0 to remove and 58 not upgraded.
Need to get 697 kB of archives.
After this operation, 2,360 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 bcftools amd64 1.13-1 [697 kB]
Fetched 697 kB in 1s (550 kB/s)
Selecting previously unselected package bcftools.
(Reading database ... 119692 files and directories currently installed.)
Preparing to unpack .../bcftools_1.13-1_amd64.deb ...
Unpacking bcftools (1.13-1) ...
Setting up bcftools (1.13-1) ...
Processing triggers for man-db (2.10.2-1) ...

checking bcf tools installation is correctly done or not

In [ ]:
!bcftools --version

bcftools 1.13
Using htslib 1.13+ds
Copyright (C) 2021 Genome Research Ltd.
License Expat: The MIT/Expat license
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.


statitics of raw-variants.vcf

In [ ]:
!bcftools stats raw_variants.vcf > vcf_stats.txt
!head vcf_stats.txt

# This file was produced by bcftools stats (1.13+htslib-1.13+ds) and can be plotted using plot-vcfstats.
# The command line was:	bcftools stats  raw_variants.vcf
#
# Definition of sets:
# ID	[2]id	[3]tab-separated file names
ID	0	raw_variants.vcf
# SN, Summary numbers:
#   number of records   .. number of data rows in the VCF
#   number of no-ALTs   .. reference-only sites, ALT is either "." or identical to REF
#   number of SNPs      .. number of rows with a SNP


Since the output only showed metadata not real results, there is some issues. checking the issues

In [ ]:
!grep "^SN" vcf_stats.txt

SN	0	number of samples:	1
SN	0	number of records:	252320
SN	0	number of no-ALTs:	0
SN	0	number of SNPs:	215092
SN	0	number of MNPs:	0
SN	0	number of indels:	37314
SN	0	number of others:	0
SN	0	number of multiallelic sites:	2010
SN	0	number of multiallelic SNP sites:	79


the variant count is low. so need to check further

In [ ]:
!samtools idxstats dedup_rg.bam | head

1	248956422	6321325	773
10	133797422	2564325	414
11	135086622	3408515	432
12	133275309	3060603	412
13	114364328	1133269	230
14	107043718	1875441	252
15	101991189	2304187	299
16	90338345	2873330	326
17	83257441	3624882	380
18	80373285	1012067	158


reads are distributed across chromosomes but variant count is low

In [ ]:
!samtools depth dedup_rg.bam | awk '{sum+=$3} END { print "Average depth = ",sum/NR}'

Average depth =  21.7549


Average depth is okay

creating folder to save raw_variants.vcf in drive

In [ ]:
!mkdir -p /content/drive/MyDrive/variant_results

In [ ]:
!cp raw_variants.vcf /content/drive/MyDrive/variant_results/

checking variant calling results further

In [ ]:
!grep -v "^#" raw_variants.vcf | cut -f1 | sort | uniq

1
10
11
12
13
14
15
16
17
18
19
2
20
21
22
3
4
5
6
7
8
9
GL000008.2
GL000009.2
GL000194.1
GL000195.1
GL000205.2
GL000213.1
GL000214.1
GL000216.2
GL000218.1
GL000219.1
GL000220.1
GL000221.1
GL000224.1
GL000225.1
KI270310.1
KI270330.1
KI270336.1
KI270337.1
KI270429.1
KI270435.1
KI270438.1
KI270442.1
KI270466.1
KI270467.1
KI270507.1
KI270517.1
KI270519.1
KI270538.1
KI270583.1
KI270589.1
KI270590.1
KI270593.1
KI270706.1
KI270708.1
KI270709.1
KI270710.1
KI270711.1
KI270712.1
KI270713.1
KI270714.1
KI270718.1
KI270719.1
KI270720.1
KI270721.1
KI270722.1
KI270723.1
KI270724.1
KI270725.1
KI270726.1
KI270727.1
KI270728.1
KI270729.1
KI270730.1
KI270731.1
KI270732.1
KI270733.1
KI270734.1
KI270735.1
KI270736.1
KI270737.1
KI270738.1
KI270742.1
KI270743.1
KI270744.1
KI270745.1
KI270746.1
KI270747.1
KI270748.1
KI270749.1
KI270750.1
KI270751.1
KI270753.1
KI270756.1
KI270757.1
MT
X
Y


So, results suggest that variant calling is not region restricted. Problem may be due to: I ran Haplotype caller without GVCF mode and Genotyping which may cause conservative calls. So, need to run again.

checking all requirements are fulfilled

In [ ]:
!samtools quickcheck dedup_rg.bam

Reference consistency check

In [ ]:
!samtools view -H dedup_rg.bam | grep "@SQ" | head

@SQ	SN:1	LN:248956422
@SQ	SN:10	LN:133797422
@SQ	SN:11	LN:135086622
@SQ	SN:12	LN:133275309
@SQ	SN:13	LN:114364328
@SQ	SN:14	LN:107043718
@SQ	SN:15	LN:101991189
@SQ	SN:16	LN:90338345
@SQ	SN:17	LN:83257441
@SQ	SN:18	LN:80373285


In [ ]:
!head Homo_sapiens.GRCh38.dna.primary_assembly.dict

@HD	VN:1.6
@SQ	SN:1	LN:248956422	M5:2648ae1bacce4ec4b6cf337dcae37816	UR:file:/content/Homo_sapiens.GRCh38.dna.primary_assembly.fa
@SQ	SN:10	LN:133797422	M5:907112d17fcb73bcab1ed1c72b97ce68	UR:file:/content/Homo_sapiens.GRCh38.dna.primary_assembly.fa
@SQ	SN:11	LN:135086622	M5:1511375dc2dd1b633af8cf439ae90cec	UR:file:/content/Homo_sapiens.GRCh38.dna.primary_assembly.fa
@SQ	SN:12	LN:133275309	M5:e81e16d3f44337034695a29b97708fce	UR:file:/content/Homo_sapiens.GRCh38.dna.primary_assembly.fa
@SQ	SN:13	LN:114364328	M5:17dab79b963ccd8e7377cef59a54fe1c	UR:file:/content/Homo_sapiens.GRCh38.dna.primary_assembly.fa
@SQ	SN:14	LN:107043718	M5:acbd9552c059d9b403e75ed26c1ce5bc	UR:file:/content/Homo_sapiens.GRCh38.dna.primary_assembly.fa
@SQ	SN:15	LN:101991189	M5:f036bd11158407596ca6bf3581454706	UR:file:/content/Homo_sapiens.GRCh38.dna.primary_assembly.fa
@SQ	SN:16	LN:90338345	M5:24e7cabfba3548a2bb4dff582b9ee870	UR:file:/content/Homo_sapiens.GRCh38.dna.primary_assembly.fa
@SQ	SN:17	LN:83257441	M5:a8499c

chromosome numbers are matching in dedup_rg and refe.genome dictionary.

checking test write permission

In [ ]:
!touch test_file && rm test_file

In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   34G   74G  32% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G     0  5.8G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1       114G   35G   80G  31% /kaggle/input
tmpfs           6.4G  512K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive            15G   14G  1.1G  94% /content/drive


Running variant calling in GVCF mode

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar HaplotypeCaller \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-I dedup_rg.bam \
-O sample.g.vcf.gz \
-ERC GVCF \
--native-pair-hmm-threads 2

11:20:56.037 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
11:20:56.152 INFO  HaplotypeCaller - ------------------------------------------------------------
11:20:56.159 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.4.0.0
11:20:56.159 INFO  HaplotypeCaller - For support and documentation go to https://software.broadinstitute.org/gatk/
11:20:56.159 INFO  HaplotypeCaller - Executing as root@75fd35667b0c on Linux v6.6.113+ amd64
11:20:56.159 INFO  HaplotypeCaller - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
11:20:56.160 INFO  HaplotypeCaller - Start Date/Time: April 28, 2026 at 11:20:55 AM UTC
11:20:56.160 INFO  HaplotypeCaller - ------------------------------------------------------------
11:20:56.160 INFO  HaplotypeCaller - ------------------------------------------------------------
11:20:56.161 INFO  HaplotypeCaller - HTSJDK 

checking results file existence

In [ ]:
!ls -lh sample.g.vcf.gz

-rw-r--r-- 1 root root 380M Apr 28 16:08 sample.g.vcf.gz


checking whether file is valid or not

In [ ]:
!bcftools view -h sample.g.vcf.gz | head

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele not already represented at this location by REF and ALT">
##FILTER=<ID=LowQual,Description="Low quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=MIN_DP,Number=1,Type=Integer,Description="Minimum DP observed within the GVCF block">
##FORMAT=<ID=PGT,Number=1,Type=String,Description="Physical phasing haplotype information, describing how the alternate alleles are phased in relation to one another; will always be heterozygous and is not intended to describe called alleles">


checking contigs in file

In [ ]:
!bcftools view -h sample.g.vcf.gz | grep "^##contig" | wc -l

194


total counts

In [ ]:
!bcftools view -H sample.g.vcf.gz | wc -l

21823938


checking structure of gvcf file

In [ ]:
!bcftools view sample.g.vcf.gz | head -n 50

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele not already represented at this location by REF and ALT">
##FILTER=<ID=LowQual,Description="Low quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=MIN_DP,Number=1,Type=Integer,Description="Minimum DP observed within the GVCF block">
##FORMAT=<ID=PGT,Number=1,Type=String,Description="Physical phasing haplotype information, describing how the alternate alleles are phased in relation to one another; will always be heterozygous and is not intended to describe called alleles">
##FORMAT=<ID=PID,Number=1,Type

structure is correct. so downloading compressed gvcf files

In [ ]:
from google.colab import files
files.download('sample.g.vcf.gz')
files.download('sample.g.vcf.gz.tbi')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

creating folder to save

In [ ]:
!mkdir -p /content/drive/MyDrive/variant_gvcf

In [ ]:
!cp sample.g.vcf.gz /content/drive/MyDrive/variant_gvcf/
!cp sample.g.vcf.gz.tbi /content/drive/MyDrive/variant_gvcf/

In [ ]:
!ls -lh /content/drive/MyDrive/variant_gvcf/

total 383M
-rw------- 1 root root 380M Apr 28 16:42 sample.g.vcf.gz
-rw------- 1 root root 3.0M Apr 28 16:42 sample.g.vcf.gz.tbi


In [ ]:
!ls -lh Homo_sapiens.GRCh38.dna.primary_assembly.fa*
!ls -lh sample.g.vcf.gz sample.g.vcf.gz.tbi

-rw------- 1 root root 3.0G Apr 28 11:01 Homo_sapiens.GRCh38.dna.primary_assembly.fa
-rw------- 1 root root 6.3K Apr 28 11:01 Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai
-rw-r--r-- 1 root root 380M Apr 28 16:08 sample.g.vcf.gz
-rw-r--r-- 1 root root 3.0M Apr 28 16:08 sample.g.vcf.gz.tbi


Genotyping

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar GenotypeGVCFs \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-V sample.g.vcf.gz \
-O final_variants.vcf \
--native-pair-hmm-threads 2

USAGE: GenotypeGVCFs [arguments]

Perform joint genotyping on a single-sample GVCF from HaplotypeCaller or a multi-sample GVCF from CombineGVCFs or
GenomicsDBImport
Version:4.4.0.0


Required Arguments:

--output,-O <GATKPath>        File to which variants should be written  Required. 

--reference,-R <GATKPath>     Reference sequence file  Required. 

--variant,-V <String>         A VCF file containing variants  Required. 


Optional Arguments:

--add-output-sam-program-record <Boolean>
                              If true, adds a PG tag to created SAM/BAM/CRAM files.  Default value: true. Possible
                              values: {true, false} 

--add-output-vcf-command-line <Boolean>
                              If true, adds a command line header line to created VCF files.  Default value: true.
                              Possible values: {true, false} 

--allele-fraction-error <Double>
                              Margin of error in allele fraction to consider a somatic 

removing the native pair parameter and running again

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar GenotypeGVCFs \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-V sample.g.vcf.gz \
-O final_variants.vcf

17:11:04.129 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
17:11:04.247 INFO  GenotypeGVCFs - ------------------------------------------------------------
17:11:04.255 INFO  GenotypeGVCFs - The Genome Analysis Toolkit (GATK) v4.4.0.0
17:11:04.255 INFO  GenotypeGVCFs - For support and documentation go to https://software.broadinstitute.org/gatk/
17:11:04.255 INFO  GenotypeGVCFs - Executing as root@75fd35667b0c on Linux v6.6.113+ amd64
17:11:04.256 INFO  GenotypeGVCFs - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
17:11:04.256 INFO  GenotypeGVCFs - Start Date/Time: April 28, 2026 at 5:11:04 PM UTC
17:11:04.256 INFO  GenotypeGVCFs - ------------------------------------------------------------
17:11:04.256 INFO  GenotypeGVCFs - ------------------------------------------------------------
17:11:04.258 INFO  GenotypeGVCFs - HTSJDK Version: 3.0.5
17:1

Downloading final results

In [ ]:
from google.colab import files
files.download('final_variants.vcf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

checking size

In [ ]:
!du -h final_variants.vcf

48M	final_variants.vcf


total count

In [ ]:
!bcftools view -H final_variants.vcf | wc -l

saving the results

In [ ]:
!cp final_variants.vcf /content/drive/MyDrive/variant_gvcf/

In [ ]:
!ls -lh /content/drive/MyDrive/variant_gvcf/final_variants.vcf

-rw------- 1 root root 48M Apr 28 17:41 /content/drive/MyDrive/variant_gvcf/final_variants.vcf


compressing for downstream analysis

In [ ]:
!bgzip final_variants.vcf
!tabix -p vcf final_variants.vcf.gz

/bin/bash: line 1: bgzip: command not found
/bin/bash: line 1: tabix: command not found


installing tabix

In [ ]:
!apt-get update
!apt-get install -y tabix

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tabix
0 upgraded, 1 newly installed, 0 to remo

compressing

In [ ]:
!bgzip final_variants.vcf
!tabix -p vcf final_variants.vcf.gz

In [ ]:
!ls -lh final_variants.vcf.gz*

-rw-r--r-- 1 root root 7.7M Apr 28 17:53 final_variants.vcf.gz
-rw-r--r-- 1 root root 589K Apr 28 17:53 final_variants.vcf.gz.tbi


saving to drive folder

In [ ]:
!cp final_variants.vcf.gz* /content/drive/MyDrive/variant_gvcf/

SNP split

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar SelectVariants \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-V final_variants.vcf.gz \
--select-type-to-include SNP \
-O raw_snps.vcf

17:59:30.557 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
17:59:30.676 INFO  SelectVariants - ------------------------------------------------------------
17:59:30.687 INFO  SelectVariants - The Genome Analysis Toolkit (GATK) v4.4.0.0
17:59:30.687 INFO  SelectVariants - For support and documentation go to https://software.broadinstitute.org/gatk/
17:59:30.688 INFO  SelectVariants - Executing as root@75fd35667b0c on Linux v6.6.113+ amd64
17:59:30.688 INFO  SelectVariants - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
17:59:30.689 INFO  SelectVariants - Start Date/Time: April 28, 2026 at 5:59:30 PM UTC
17:59:30.689 INFO  SelectVariants - ------------------------------------------------------------
17:59:30.689 INFO  SelectVariants - ------------------------------------------------------------
17:59:30.691 INFO  SelectVariants - HTSJDK Version: 3

Indel split

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar SelectVariants \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-V final_variants.vcf.gz \
--select-type-to-include INDEL \
-O raw_indels.vcf

18:05:51.588 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
18:05:51.716 INFO  SelectVariants - ------------------------------------------------------------
18:05:51.724 INFO  SelectVariants - The Genome Analysis Toolkit (GATK) v4.4.0.0
18:05:51.724 INFO  SelectVariants - For support and documentation go to https://software.broadinstitute.org/gatk/
18:05:51.724 INFO  SelectVariants - Executing as root@75fd35667b0c on Linux v6.6.113+ amd64
18:05:51.724 INFO  SelectVariants - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
18:05:51.725 INFO  SelectVariants - Start Date/Time: April 28, 2026 at 6:05:51 PM UTC
18:05:51.725 INFO  SelectVariants - ------------------------------------------------------------
18:05:51.725 INFO  SelectVariants - ------------------------------------------------------------
18:05:51.727 INFO  SelectVariants - HTSJDK Version: 3

In [ ]:
!ls -lh

total 6.9G
-rw------- 1 root root 2.9G Apr 28 11:08 dedup_rg.bam
-rw------- 1 root root 4.1M Apr 28 11:08 dedup_rg.bam.bai
drwx------ 5 root root 4.0K Apr 28 11:00 drive
-rw-r--r-- 1 root root 7.7M Apr 28 17:53 final_variants.vcf.gz
-rw-r--r-- 1 root root 589K Apr 28 17:53 final_variants.vcf.gz.tbi
-rw-r--r-- 1 root root 4.8M Apr 28 17:15 final_variants.vcf.idx
drwxr-xr-x 4 root root 4.0K Mar 16  2023 gatk-4.4.0.0
-rw-r--r-- 1 root root 613M Mar 16  2023 gatk-4.4.0.0.zip
-rw------- 1 root root  24K Apr 28 11:00 Homo_sapiens.GRCh38.dna.primary_assembly.dict
-rw------- 1 root root 3.0G Apr 28 11:01 Homo_sapiens.GRCh38.dna.primary_assembly.fa
-rw------- 1 root root 6.3K Apr 28 11:01 Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai
-rw-r--r-- 1 root root 7.3M Apr 28 18:05 raw_indels.vcf
-rw-r--r-- 1 root root  35K Apr 28 18:05 raw_indels.vcf.idx
-rw-r--r-- 1 root root  41M Apr 28 17:59 raw_snps.vcf
-rw-r--r-- 1 root root 4.2M Apr 28 17:59 raw_snps.vcf.idx
drwxr-xr-x 1 root root 4.0K Apr 16 

variant filtration - SNP

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar VariantFiltration \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-V raw_snps.vcf \
-O filtered_snps.vcf \
--filter-name "SNP_filter" \
--filter-expression "QD < 2.0 || FS > 60.0 || MQ < 40.0 || SOR > 3.0"

18:16:46.227 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
18:16:46.312 INFO  VariantFiltration - ------------------------------------------------------------
18:16:46.320 INFO  VariantFiltration - The Genome Analysis Toolkit (GATK) v4.4.0.0
18:16:46.323 INFO  VariantFiltration - For support and documentation go to https://software.broadinstitute.org/gatk/
18:16:46.323 INFO  VariantFiltration - Executing as root@75fd35667b0c on Linux v6.6.113+ amd64
18:16:46.324 INFO  VariantFiltration - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
18:16:46.326 INFO  VariantFiltration - Start Date/Time: April 28, 2026 at 6:16:46 PM UTC
18:16:46.326 INFO  VariantFiltration - ------------------------------------------------------------
18:16:46.326 INFO  VariantFiltration - ------------------------------------------------------------
18:16:46.327 INFO  VariantFil

variant filtration -indel

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar VariantFiltration \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-V raw_indels.vcf \
-O filtered_indels.vcf \
--filter-name "INDEL_filter" \
--filter-expression "QD < 2.0 || FS > 200.0 || SOR > 10.0"

18:24:39.445 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
18:24:39.551 INFO  VariantFiltration - ------------------------------------------------------------
18:24:39.557 INFO  VariantFiltration - The Genome Analysis Toolkit (GATK) v4.4.0.0
18:24:39.557 INFO  VariantFiltration - For support and documentation go to https://software.broadinstitute.org/gatk/
18:24:39.557 INFO  VariantFiltration - Executing as root@75fd35667b0c on Linux v6.6.113+ amd64
18:24:39.558 INFO  VariantFiltration - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
18:24:39.558 INFO  VariantFiltration - Start Date/Time: April 28, 2026 at 6:24:39 PM UTC
18:24:39.558 INFO  VariantFiltration - ------------------------------------------------------------
18:24:39.558 INFO  VariantFiltration - ------------------------------------------------------------
18:24:39.560 INFO  VariantFil

In [ ]:
!ls -lh filtered_*.vcf

-rw-r--r-- 1 root root 7.4M Apr 28 18:24 filtered_indels.vcf
-rw-r--r-- 1 root root  41M Apr 28 18:16 filtered_snps.vcf


SNP-pass numbers of variants

In [ ]:
!bcftools view -f PASS filtered_snps.vcf | grep -v "^#" | wc -l

172260


outputs saved into vcf files

In [ ]:
!bcftools view -f PASS filtered_snps.vcf > final_snps_pass.vcf


In [ ]:
!bcftools view -f PASS filtered_indels.vcf > final_indels_pass.vcf

indels pass- numbers of variants

In [ ]:
!grep -v "^#" final_indels_pass.vcf | wc -l

36307


In [ ]:
!bcftools view filtered_indels.vcf | grep -v "^#" | cut -f7 | sort | uniq -c

    898 INDEL_filter
  36307 PASS


Since indel filtration is lenient, I am applying more stringent filtration

tightening filters

In [ ]:
!java -jar /content/gatk-4.4.0.0/gatk-package-4.4.0.0-local.jar VariantFiltration \
-R Homo_sapiens.GRCh38.dna.primary_assembly.fa \
-V raw_indels.vcf \
-O filtered_indels_strict.vcf \
--filter-name "INDEL_filter" \
--filter-expression "QD < 2.0 || FS > 100.0 || SOR > 5.0 || ReadPosRankSum < -20.0"

Streaming output truncated to the last 5000 lines.
18:36:50.683 WARN  JexlEngine - ![39,53]: 'QD < 2.0 || FS > 100.0 || SOR > 5.0 || ReadPosRankSum < -20.0;' undefined variable ReadPosRankSum
18:36:50.684 WARN  JexlEngine - ![39,53]: 'QD < 2.0 || FS > 100.0 || SOR > 5.0 || ReadPosRankSum < -20.0;' undefined variable ReadPosRankSum
18:36:50.684 WARN  JexlEngine - ![39,53]: 'QD < 2.0 || FS > 100.0 || SOR > 5.0 || ReadPosRankSum < -20.0;' undefined variable ReadPosRankSum
18:36:50.684 WARN  JexlEngine - ![39,53]: 'QD < 2.0 || FS > 100.0 || SOR > 5.0 || ReadPosRankSum < -20.0;' undefined variable ReadPosRankSum
18:36:50.684 WARN  JexlEngine - ![39,53]: 'QD < 2.0 || FS > 100.0 || SOR > 5.0 || ReadPosRankSum < -20.0;' undefined variable ReadPosRankSum
18:36:50.684 WARN  JexlEngine - ![39,53]: 'QD < 2.0 || FS > 100.0 || SOR > 5.0 || ReadPosRankSum < -20.0;' undefined variable ReadPosRankSum
18:36:50.684 WARN  JexlEngine - ![39,53]: 'QD < 2.0 || FS > 100.0 || SOR > 5.0 || ReadPosRankSum < -20.

In [ ]:
!bcftools view -f PASS filtered_indels_strict.vcf | grep -v "^#" | wc -l

34993


In [ ]:
# Before filtering
!grep -v "^#" raw_snps.vcf | wc -l
!grep -v "^#" raw_indels.vcf | wc -l

# After filtering (PASS only)
!bcftools view -f PASS filtered_snps.vcf | grep -v "^#" | wc -l
!bcftools view -f PASS filtered_indels.vcf | grep -v "^#" | wc -l

merging both pass vcf files

In [ ]:
!bcftools concat final_snps_pass.vcf final_indels_pass.vcf -o final_filtered_variants.vcf

Checking the headers and starting positions of 2 files
Concatenating final_snps_pass.vcf	0.198627 seconds
Concatenating final_indels_pass.vcf
The chromosome block 1 is not contiguous, consider running with -a.


concat is not the right command. i will use merge command

In [ ]:
!bcftools merge final_snps_pass.vcf final_indels_pass.vcf -o final_filtered_variants.vcf

Failed to open final_snps_pass.vcf: not compressed with bgzip


compressing with bgzip

In [ ]:
!bgzip final_snps_pass.vcf
!bgzip final_indels_pass.vcf

In [ ]:
!tabix -p vcf final_snps_pass.vcf.gz
!tabix -p vcf final_indels_pass.vcf.gz

In [ ]:
!bcftools merge final_snps_pass.vcf.gz final_indels_pass.vcf.gz -o final_filtered_variants.vcf

Error: Duplicate sample names (sample1), use --force-samples to proceed anyway.


merging both compressed pass files

In [ ]:
!bcftools concat -a final_snps_pass.vcf.gz final_indels_pass.vcf.gz -o final_filtered_variants.vcf

Checking the headers and starting positions of 2 files


compressing final filtered files

In [ ]:
!bgzip final_filtered_variants.vcf
!tabix -p vcf final_filtered_variants.vcf.gz

In [ ]:
!ls -lh final_*pass*

-rw-r--r-- 1 root root 1.3M Apr 28 18:57 final_indels_pass.vcf.gz
-rw-r--r-- 1 root root 236K Apr 28 18:57 final_indels_pass.vcf.gz.tbi
-rw-r--r-- 1 root root 4.9M Apr 28 18:57 final_snps_pass.vcf.gz
-rw-r--r-- 1 root root 521K Apr 28 18:57 final_snps_pass.vcf.gz.tbi


checking stats and counts

In [ ]:
!ls -lh final_filtered_variants.vcf.gz
!bcftools view -H final_filtered_variants.vcf.gz | wc -l

-rw-r--r-- 1 root root 6.2M Apr 28 19:02 final_filtered_variants.vcf.gz
208567


creating directory to save in drive

In [ ]:
!mkdir -p /content/drive/MyDrive/variant_filter

In [ ]:
!cp final_filtered_variants.vcf.gz* /content/drive/MyDrive/variant_filter/
!cp sample.g.vcf.gz* /content/drive/MyDrive/variant_filter/

downloading final filtered files

In [ ]:
from google.colab import files
files.download('final_filtered_variants.vcf.gz')
files.download('final_filtered_variants.vcf.gz.tbi')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls -lh /content/drive/MyDrive/variant_dedup/

total 5.8G
-rw------- 1 root root 2.9G Apr 26 13:24 dedup.bam
-rw------- 1 root root 4.1M Apr 26 13:24 dedup.bam.bai
-rw------- 1 root root 3.3K Apr 26 13:24 dedup_metrics.txt
-rw------- 1 root root 2.9G Apr 26 18:36 dedup_rg.bam
-rw------- 1 root root 4.1M Apr 26 18:37 dedup_rg.bam.bai


removing unnecessary files

In [ ]:
!rm dedup.bam
!rm dedup.bam.bai

In [ ]:
!cp /content/drive/MyDrive/variant_dedup/dedup_rg.bam .
!cp /content/drive/MyDrive/variant_dedup/dedup_rg.bam.bai .

In [ ]:
!mkdir -p /content/drive/MyDrive/reference_genome

In [ ]:
!ls -lh /content/drive/MyDrive/reference_genome/

In [ ]:
!ls -lh /content/drive/MyDrive/

total 3.2G
-rw------- 1 root root 420K Aug 30  2016  9EefDPKA892VBWAF20Kt7SE9.jpg
-rw------- 1 root root 4.1M Apr 26 12:30  aligned_sorted.bam.bai
drwx------ 2 root root 4.0K Apr  9 09:15 'Colab Notebooks'
-rw------- 1 root root  175 Nov  1  2023 'Micro topics in Modern History.gdoc'
drwx------ 2 root root 4.0K Apr 27 15:10  reference_genome
-r-------- 1 root root  175 Oct 26  2023 'Socio religious reform 1.gslides'
drwx------ 2 root root 4.0K Apr 26 13:22  variant_dedup
drwx------ 2 root root 4.0K Apr 28 18:52  variant_filter
drwx------ 2 root root 4.0K Apr 28 16:40  variant_gvcf
drwx------ 2 root root 4.0K Apr 27 19:03  variant_results
-rw------- 1 root root  68M Oct 27  2023  video1029288025.mp4
-rw------- 1 root root  78M Nov 21  2023  video1092694532.mp4
-rw------- 1 root root  64M Nov 13  2023  video1123749661.mp4
-rw------- 1 root root  66M Nov  1  2023  video1141827647.mp4
-rw------- 1 root root  84M Nov 21  2023  video1157375517.mp4
-rw------- 1 root root  72M Nov  9  2023  vi

In [ ]:
!cp /content/drive/MyDrive/variant_filter/final_filtered_variants.vcf.gz .
!cp /content/drive/MyDrive/variant_filter/final_filtered_variants.vcf.gz.tbi .

!cp /content/drive/MyDrive/reference_genome/Homo_sapiens.GRCh38.dna.primary_assembly.fa .
!cp /content/drive/MyDrive/reference_genome/Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai .
!cp /content/drive/MyDrive/reference_genome/Homo_sapiens.GRCh38.dna.primary_assembly.dict .

In [ ]:
!ls -lh

total 3.6G
drwx------ 5 root root 4.0K Apr 29 13:46 drive
-rw------- 1 root root 6.2M Apr 29 14:25 final_filtered_variants.vcf.gz
-rw------- 1 root root 569K Apr 29 14:25 final_filtered_variants.vcf.gz.tbi
drwxr-xr-x 4 root root 4.0K Mar 16  2023 gatk-4.4.0.0
-rw-r--r-- 1 root root 613M Mar 16  2023 gatk-4.4.0.0.zip
-rw------- 1 root root  24K Apr 29 14:26 Homo_sapiens.GRCh38.dna.primary_assembly.dict
-rw------- 1 root root 3.0G Apr 29 14:26 Homo_sapiens.GRCh38.dna.primary_assembly.fa
-rw------- 1 root root 6.3K Apr 29 14:26 Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data


In [ ]:
!bcftools view -H final_filtered_variants.vcf.gz | wc -l

208567


In [ ]:
!wget https://snpeff.blob.core.windows.net/versions/snpEff_latest_core.zip
!unzip snpEff_latest_core.zip

--2026-04-29 15:02:22--  https://snpeff.blob.core.windows.net/versions/snpEff_latest_core.zip
Resolving snpeff.blob.core.windows.net (snpeff.blob.core.windows.net)... failed: Name or service not known.
wget: unable to resolve host address ‘snpeff.blob.core.windows.net’
unzip:  cannot find or open snpEff_latest_core.zip, snpEff_latest_core.zip.zip or snpEff_latest_core.zip.ZIP.


In [ ]:
!wget https://github.com/pcingola/SnpEff/releases/latest/download/snpEff_latest_core.zip
!unzip snpEff_latest_core.zip

--2026-04-29 15:00:49--  https://github.com/pcingola/SnpEff/releases/latest/download/snpEff_latest_core.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-29 15:00:49 ERROR 404: Not Found.

unzip:  cannot find or open snpEff_latest_core.zip, snpEff_latest_core.zip.zip or snpEff_latest_core.zip.ZIP.


In [ ]:
!curl -L -o snpEff_latest_core.zip https://github.com/pcingola/SnpEff/releases/latest/download/snpEff_latest_core.zip
!unzip snpEff_latest_core.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100     9  100     9    0     0     31      0 --:--:-- --:--:-- --:--:--    31
Archive:  snpEff_latest_core.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of snpEff_latest_core.zip or
        snpEff_latest_core.zip.zip, and cannot find snpEff_latest_core.zip.ZIP, period.


Since the above commands did not work, I checked with this link and it worked

In [ ]:
!wget https://snpeff-public.s3.amazonaws.com/versions/snpEff_latest_core.zip

--2026-04-29 15:08:52--  https://snpeff-public.s3.amazonaws.com/versions/snpEff_latest_core.zip
Resolving snpeff-public.s3.amazonaws.com (snpeff-public.s3.amazonaws.com)... 52.218.176.99, 52.92.138.137, 3.5.81.2, ...
Connecting to snpeff-public.s3.amazonaws.com (snpeff-public.s3.amazonaws.com)|52.218.176.99|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66677840 (64M) [application/zip]
Saving to: ‘snpEff_latest_core.zip.1’

snpEff_latest_core. 100%[===================>]  63.59M  30.1MB/s    in 2.1s    

2026-04-29 15:08:55 (30.1 MB/s) - ‘snpEff_latest_core.zip.1’ saved [66677840/66677840]



In [ ]:
!ls -lh snpEff_latest_core.zip.1

-rw-r--r-- 1 root root 64M Feb 23 12:51 snpEff_latest_core.zip.1


In [ ]:
!stat snpEff_latest_core.zip.1

  File: snpEff_latest_core.zip.1
  Size: 66677840  	Blocks: 130232     IO Block: 4096   regular file
Device: 36h/54d	Inode: 1973893     Links: 1
Access: (0644/-rw-r--r--)  Uid: (    0/    root)   Gid: (    0/    root)
Access: 2026-04-29 15:08:59.174904403 +0000
Modify: 2026-02-23 12:51:23.000000000 +0000
Change: 2026-04-29 15:08:55.236867007 +0000
 Birth: 2026-04-29 15:08:53.126846969 +0000


previously snpeff file is corrupted. I am renaming this

In [ ]:
!mv snpEff_latest_core.zip.1 snpEff_latest_core.zip

unzipping snpEff

In [ ]:
!unzip snpEff_latest_core.zip

Archive:  snpEff_latest_core.zip
   creating: snpEff/
  inflating: snpEff/LICENSE.md       
  inflating: snpEff/snpEff.jar       
  inflating: snpEff/SnpSift.jar      
   creating: snpEff/.claude/
  inflating: snpEff/.claude/settings.local.json  
   creating: snpEff/.claude/skills/
   creating: snpEff/.claude/skills/snpeff/
   creating: snpEff/.claude/skills/snpeff/docs/
  inflating: snpEff/.claude/skills/snpeff/docs/human_genomes.md  
  inflating: snpEff/.claude/skills/snpeff/docs/build_pdb.md  
  inflating: snpEff/.claude/skills/snpeff/docs/inputoutput.md  
  inflating: snpEff/.claude/skills/snpeff/docs/build_db.md  
  inflating: snpEff/.claude/skills/snpeff/docs/examples.md  
  inflating: snpEff/.claude/skills/snpeff/docs/license.md  
  inflating: snpEff/.claude/skills/snpeff/docs/outputsummary.md  
  inflating: snpEff/.claude/skills/snpeff/docs/integration.md  
  inflating: snpEff/.claude/skills/snpeff/docs/troubleshooting.md  
  inflating: snpEff/.claude/skills/snpeff/docs/running

In [ ]:
!cd snpEff

In [ ]:
!ls snpEff

examples  galaxy      scripts	     snpEff.jar
exec	  LICENSE.md  snpEff.config  SnpSift.jar


In [ ]:
!java -jar snpEff/snpEff.jar

Error: LinkageError occurred while loading main class org.snpeff.SnpEff
	java.lang.UnsupportedClassVersionError: org/snpeff/SnpEff has been compiled by a more recent version of the Java Runtime (class file version 65.0), this version of the Java Runtime only recognizes class file versions up to 61.0


java version was not supporting, so need to update with java version 21


In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y openjdk-21-jdk

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libgtk-3-0 libgtk-3-bin libgtk-3-common

configuring

In [ ]:
!sudo update-alternatives --config java

There are 3 choices for the alternative java (providing /usr/bin/java).

  Selection    Path                                         Priority   Status
------------------------------------------------------------
* 0            /usr/lib/jvm/java-21-openjdk-amd64/bin/java   2111      auto mode
  1            /usr/lib/jvm/java-11-openjdk-amd64/bin/java   1111      manual mode
  2            /usr/lib/jvm/java-17-openjdk-amd64/bin/java   1711      manual mode
  3            /usr/lib/jvm/java-21-openjdk-amd64/bin/java   2111      manual mode

Press <enter> to keep the current choice[*], or type selection number: 3


version check

In [ ]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)


Downloading annotation database

In [ ]:
!java -jar snpEff/snpEff.jar download GRCh38.99

[0.006s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.006s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


this is only warning not an error

checking the download

In [ ]:
!java -jar snpEff/snpEff.jar databases | grep GRCh38

GRCh38.115                                                  	Homo_sapiens                                                	          	                              	[https://snpeff-public.s3.amazonaws.com/databases/v5_4/snpEff_v5_4_GRCh38.115.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_3/snpEff_v5_3_GRCh38.115.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_2/snpEff_v5_2_GRCh38.115.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_1/snpEff_v5_1_GRCh38.115.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_0/snpEff_v5_0_GRCh38.115.zip]
GRCh38.86                                                   	GRCh38.86                                                   	          	                              	[https://snpeff-public.s3.amazonaws.com/databases/v5_4/snpEff_v5_4_GRCh38.86.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_3/snpEff_v5_3_GRCh38.86.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_2/snpEff_v5_2_GRCh38.86.zip, https://snpeff-pu

checking whether snpEff setup is correct for annoation.

In [ ]:
!ls snpEff
!ls snpEff/data

data	  exec	  LICENSE.md  snpEff.config  SnpSift.jar
examples  galaxy  scripts     snpEff.jar
GRCh38.99


Annotation with GRCh38.99

In [ ]:
!java -Xmx4g -jar snpEff/snpEff.jar GRCh38.99 final_filtered_variants.vcf.gz > annotated.vcf

In [ ]:
!bgzip annotated.vcf
!tabix -p vcf annotated.vcf.gz

[E::get_intv] Failed to parse TBX_VCF, was wrong -p [type] used?
The offending line was: "[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate"
[E::get_intv] Failed to parse TBX_VCF, was wrong -p [type] used?
The offending line was: "[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate"


file is corrupted or having log text mixed into vcf

Re-running annotation step

In [ ]:
!java -Xmx4g -jar snpEff/snpEff.jar GRCh38.99 final_filtered_variants.vcf.gz \
> annotated_clean.vcf 2> snpeff.log

In [ ]:
!head annotated_clean.vcf

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele not already represented at this location by REF and ALT">
##FILTER=<ID=LowQual,Description="Low quality">
##FILTER=<ID=SNP_filter,Description="QD < 2.0 || FS > 60.0 || MQ < 40.0 || SOR > 3.0">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">


The same issue happened, log outputs are getting mixed into vcf

So, explicitly, removing the mixed lines

In [ ]:
!grep -v "^\[" annotated_clean.vcf > annotated_fixed.vcf

checking the cleaned output

In [ ]:
!head annotated_fixed.vcf

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele not already represented at this location by REF and ALT">
##FILTER=<ID=LowQual,Description="Low quality">
##FILTER=<ID=SNP_filter,Description="QD < 2.0 || FS > 60.0 || MQ < 40.0 || SOR > 3.0">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=MIN_DP,Number=1,Type=Integer,Description="Minimum DP observed within the GVCF block">


checking file presence

In [ ]:
!ls -lh

total 4.3G
-rw-r--r-- 1 root root 215M Apr 29 17:52 annotated_clean.vcf
-rw-r--r-- 1 root root 215M Apr 29 17:59 annotated_fixed.vcf
-rw-r--r-- 1 root root 215M Apr 29 17:41 annotated.vcf
-rw-r--r-- 1 root root  25M Apr 29 17:27 annotated.vcf.gz
-rw-r--r-- 1 root root 603K Apr 29 17:27 annotated.vcf.gz.tbi
drwx------ 5 root root 4.0K Apr 29 13:46 drive
-rw------- 1 root root 6.2M Apr 29 14:25 final_filtered_variants.vcf.gz
-rw------- 1 root root 569K Apr 29 14:25 final_filtered_variants.vcf.gz.tbi
drwxr-xr-x 4 root root 4.0K Mar 16  2023 gatk-4.4.0.0
-rw-r--r-- 1 root root 613M Mar 16  2023 gatk-4.4.0.0.zip
-rw------- 1 root root  24K Apr 29 14:26 Homo_sapiens.GRCh38.dna.primary_assembly.dict
-rw------- 1 root root 3.0G Apr 29 14:26 Homo_sapiens.GRCh38.dna.primary_assembly.fa
-rw------- 1 root root 6.3K Apr 29 14:26 Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data
drwxr-xr-x 8 root root 4.0K Apr 29 15:40 snpEff
-rw-r--r-- 1 root root 

compressing the final cleaned annotation file

In [ ]:
!bgzip -f annotated_fixed.vcf

In [ ]:
!tabix -f -p vcf annotated_fixed.vcf.gz

checking annotation happened correctly or not

In [ ]:
!bcftools view -h annotated_fixed.vcf.gz | grep ANN

##INFO=<ID=ANN,Number=.,Type=String,Description="Functional annotations: 'Allele | Annotation | Annotation_Impact | Gene_Name | Gene_ID | Feature_Type | Feature_ID | Transcript_BioType | Rank | HGVS.c | HGVS.p | cDNA.pos / cDNA.length | CDS.pos / CDS.length | AA.pos / AA.length | Distance | ERRORS / WARNINGS / INFO'">


In [ ]:
!bcftools view -H annotated_fixed.vcf.gz | wc -l

208567


removing old corrupted files

In [ ]:
!rm annotated.vcf
!rm annotated_clean.vcf

extracting key annotation fields

In [ ]:
!bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%ANN\n' annotated_fixed.vcf.gz > ann_raw.tsv

In [ ]:
from google.colab import files
files.download('annotated_fixed.vcf.gz')
files.download('annotated_fixed.vcf.gz.tbi')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

creating directory to save output

In [ ]:
!mkdir -p /content/drive/MyDrive/WES_Project_Output

copying to drive

In [ ]:
!cp annotated_fixed.vcf.gz /content/drive/MyDrive/WES_Project_Output/
!cp annotated_fixed.vcf.gz.tbi /content/drive/MyDrive/WES_Project_Output/

In [ ]:
!ls -lh /content/drive/MyDrive/WES_Project_Output/

total 26M
-rw------- 1 root root  25M Apr 29 18:34 annotated_fixed.vcf.gz
-rw------- 1 root root 603K Apr 29 18:34 annotated_fixed.vcf.gz.tbi


cleaning extraction with gene and impact

In [ ]:
!cut -f5 ann_raw.tsv | tr ',' '\n' | cut -d'|' -f2,3,4 > ann_parsed.tsv

adding column names

In [ ]:
!echo -e "Effect\tImpact\tGene" | cat - ann_parsed.tsv > ann_final.tsv

high impact variants and moderate impact variants

In [ ]:
!grep "HIGH" ann_final.tsv > high_impact.tsv

In [ ]:
!grep "MODERATE" ann_final.tsv > moderate_impact.tsv

In [ ]:
!wc -l ann_final.tsv
!wc -l high_impact.tsv
!wc -l moderate_impact.tsv

1434204 ann_final.tsv
1393 high_impact.tsv
28848 moderate_impact.tsv


saving the results

In [ ]:
!cp ann_raw.tsv /content/drive/MyDrive/WES_Project_Output/
!cp ann_final.tsv /content/drive/MyDrive/WES_Project_Output/
!cp high_impact.tsv /content/drive/MyDrive/WES_Project_Output/
!cp moderate_impact.tsv /content/drive/MyDrive/WES_Project_Output/

In [ ]:
!ls -lh /content/drive/MyDrive/WES_Project_Output/

In [ ]:
from google.colab import files
files.download('ann_raw.tsv')
files.download('ann_parsed.tsv')
files.download('ann_final.tsv')
files.download('high_impact.tsv')
files.download('moderate_impact.tsv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

because numbers above seem to contain duplicates, i need to check unique gene counts

In [ ]:
!cut -f3 ann_final.tsv | sort | uniq > unique_genes.txt

In [ ]:
!wc -l unique_genes.txt

116797 unique_genes.txt


unique high impact genes

In [ ]:
!grep HIGH ann_final.tsv | cut -f3 | sort | uniq > high_impact_genes.txt
!wc -l high_impact_genes.txt

636 high_impact_genes.txt


unique moderate impact genes

In [ ]:
!grep MODERATE ann_final.tsv | cut -f3 | sort | uniq > moderate_impact_genes.txt
!wc -l moderate_impact_genes.txt

6497 moderate_impact_genes.txt


saving outputs

In [ ]:
!cp moderate_impact_genes.txt /content/drive/MyDrive/WES_Project_Output/
!cp high_impact_genes.txt /content/drive/MyDrive/WES_Project_Output/

In [ ]:
!cp /content/drive/MyDrive/WES_Project_Output/annotated_fixed.vcf.gz .
!cp /content/drive/MyDrive/WES_Project_Output/annotated_fixed.vcf.gz.tbi .

In [ ]:
!cp /content/drive/MyDrive/WES_Project_Output/ann_final.tsv .
!cp /content/drive/MyDrive/WES_Project_Output/high_impact.tsv .
!cp /content/drive/MyDrive/WES_Project_Output/moderate_impact.tsv .
!cp /content/drive/MyDrive/WES_Project_Output/high_impact_genes.txt .
!cp /content/drive/MyDrive/WES_Project_Output/moderate_impact_genes.txt .

In [ ]:
!ls -lh

total 687M
-rw------- 1 root root  48M Apr 30 05:55 ann_final.tsv
-rw------- 1 root root  25M Apr 30 05:54 annotated_fixed.vcf.gz
-rw------- 1 root root 603K Apr 30 05:54 annotated_fixed.vcf.gz.tbi
drwx------ 5 root root 4.0K Apr 30 05:32 drive
drwxr-xr-x 4 root root 4.0K Mar 16  2023 gatk-4.4.0.0
-rw-r--r-- 1 root root 613M Mar 16  2023 gatk-4.4.0.0.zip
-rw------- 1 root root  26K Apr 30 05:55 high_impact_genes.txt
-rw------- 1 root root  55K Apr 30 05:55 high_impact.tsv
-rw------- 1 root root 217K Apr 30 05:55 moderate_impact_genes.txt
-rw------- 1 root root 941K Apr 30 05:55 moderate_impact.tsv
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data


Top mutated genes extraction

In [ ]:
!cut -f3 ann_final.tsv | sort | uniq -c | sort -nr | head -20

   5697 intron_variant|MODIFIER|ADGRG1
   2453 intron_variant|MODIFIER|ASAH1
   2184 intron_variant|MODIFIER|KCNMA1
   1893 intron_variant|MODIFIER|MUC20-OT1
   1839 intron_variant|MODIFIER|TTN-AS1
   1823 intron_variant|MODIFIER|ANK2
   1707 intron_variant|MODIFIER|PCDH15
   1548 intron_variant|MODIFIER|CACNA1C
   1310 intron_variant|MODIFIER|RBFOX1
   1304 intron_variant|MODIFIER|CSMD1
   1295 intron_variant|MODIFIER|TBCE
   1212 intron_variant|MODIFIER|SYNE1
   1178 intron_variant|MODIFIER|MAP2K3
   1172 downstream_gene_variant|MODIFIER|MUC20-OT1
   1142 intron_variant|MODIFIER|CACNA1A
   1113 intron_variant|MODIFIER|HULC
   1044 intron_variant|MODIFIER|FLG-AS1
    982 intron_variant|MODIFIER|MAPK10
    972 intron_variant|MODIFIER|SGCE
    963 intron_variant|MODIFIER|COL13A1


I am now extracting specifically from high impact genes


In [ ]:
!sort high_impact_genes.txt | uniq | head

bidirectional_gene_fusion|HIGH|AC040977.2&C17orf49
frameshift_variant|HIGH|A4GALT
frameshift_variant|HIGH|ABHD12
frameshift_variant|HIGH|ABL2
frameshift_variant|HIGH|AC084121.13
frameshift_variant|HIGH|ACTN2
frameshift_variant|HIGH|ADAMTS7
frameshift_variant|HIGH|ADAMTSL1
frameshift_variant|HIGH|AGAP3
frameshift_variant|HIGH|AGAP6


In [ ]:
!wc -l high_impact_genes.txt

636 high_impact_genes.txt


testing for known colon cancer genes

In [ ]:
!grep -E "TP53|APC|KRAS|PIK3CA|SMAD4|FBXW7|BRAF" ann_final.tsv

synonymous_variant|LOW|APCS
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
synonymous_variant|LOW|TP53BP2
synonymous_variant|LOW|TP53BP2
3_prime_UTR_variant|MODIFIER|TP53BP2
downstream_gene_variant|MODIFIER|TP53BP2
downstream_gene_variant|MODIFIER|TP53BP2
non_coding_transcript_exon_variant|MODIFIER|TP53BP2
non_coding_transcript_exon_variant|MODIFIER|TP53BP2
downstream_gene_variant|MODIFIER|TP53BP2
downstream_gene_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53BP2
intron_variant|MODIFIER|TP53B

unique cancer gene hits

In [ ]:
!grep -E "TP53|APC|KRAS|PIK3CA|SMAD4|FBXW7|BRAF" ann_final.tsv | cut -f3 | sort | uniq

3_prime_UTR_variant|MODIFIER|ANAPC1
3_prime_UTR_variant|MODIFIER|APC
3_prime_UTR_variant|MODIFIER|APCDD1
3_prime_UTR_variant|MODIFIER|KRAS
3_prime_UTR_variant|MODIFIER|MSH5-SAPCD1
3_prime_UTR_variant|MODIFIER|SNAPC3
3_prime_UTR_variant|MODIFIER|SNAPC5
3_prime_UTR_variant|MODIFIER|TP53AIP1
3_prime_UTR_variant|MODIFIER|TP53BP1
3_prime_UTR_variant|MODIFIER|TP53BP2
3_prime_UTR_variant|MODIFIER|TP53I13
3_prime_UTR_variant|MODIFIER|TP53INP1
3_prime_UTR_variant|MODIFIER|TP53RK
5_prime_UTR_variant|MODIFIER|ANAPC10
5_prime_UTR_variant|MODIFIER|ANAPC13
5_prime_UTR_variant|MODIFIER|APC2
5_prime_UTR_variant|MODIFIER|APCDD1
5_prime_UTR_variant|MODIFIER|APCDD1L
5_prime_UTR_variant|MODIFIER|SNAPC1
5_prime_UTR_variant|MODIFIER|SNAPC2
5_prime_UTR_variant|MODIFIER|TP53
5_prime_UTR_variant|MODIFIER|TP53I11
5_prime_UTR_variant|MODIFIER|TP53I13
5_prime_UTR_variant|MODIFIER|TP53RK
downstream_gene_variant|MODIFIER|ANAPC1
downstream_gene_variant|MODIFIER|ANAPC11
downstream_gene_variant|MODIFIER|ANAPC1P1
downs

top frequent moderate impact genes

In [ ]:
!grep MODERATE ann_final.tsv | cut -f3 | sort | uniq -c | sort -nr | head

    208 missense_variant|MODERATE|OBSCN
    167 missense_variant|MODERATE|TTN
    150 missense_variant|MODERATE|HLA-A
    134 missense_variant|MODERATE|MUC3A
    130 missense_variant|MODERATE|MUC4
    126 missense_variant|MODERATE|ZNF717
    120 missense_variant|MODERATE|HLA-C
    101 missense_variant|MODERATE|NEB
     64 missense_variant|MODERATE|MUC16
     64 missense_variant|MODERATE|HLA-B


Top frequent high impact genes

In [ ]:
!grep HIGH ann_final.tsv | cut -f3 | sort | uniq -c | sort -nr | head

     18 splice_acceptor_variant&intron_variant|HIGH|STAG2
     16 frameshift_variant|HIGH|FCGBP
     14 splice_acceptor_variant&intron_variant|HIGH|DGKH
     13 splice_donor_variant&intron_variant|HIGH|LIMS1
     13 splice_acceptor_variant&intron_variant|HIGH|ENDOV
     13 splice_acceptor_variant&intron_variant|HIGH|CBWD3
     12 frameshift_variant|HIGH|UBXN11
     11 splice_acceptor_variant&intron_variant|HIGH|GAPVD1
     10 splice_donor_variant&intron_variant|HIGH|TDG
     10 splice_acceptor_variant&intron_variant|HIGH|SKA3


removing known artefact genes from moderate

In [ ]:
!echo -e "TTN\nOBSCN\nNEB\nMUC3A\nMUC4\nMUC16\nZNF717" > artifact_genes.txt

In [ ]:
!grep MODERATE ann_final.tsv | grep missense | grep -v -F -f artifact_genes.txt > moderate_filtered.tsv

In [ ]:
!cut -f3 moderate_filtered.tsv | sort | uniq -c | sort -nr | head -20

    150 missense_variant|MODERATE|HLA-A
    120 missense_variant|MODERATE|HLA-C
     64 missense_variant|MODERATE|HLA-B
     60 missense_variant|MODERATE|PLEC
     57 missense_variant|MODERATE|HLA-DQB1
     57 missense_variant|MODERATE|ASAH1
     54 missense_variant|MODERATE|PIGN
     54 missense_variant|MODERATE|FLG
     51 missense_variant|MODERATE|TSBP1
     51 missense_variant|MODERATE|NBPF26
     51 missense_variant|MODERATE|AHNAK2
     50 missense_variant|MODERATE|FBN3
     48 missense_variant|MODERATE|MUC20
     48 missense_variant|MODERATE|FCGBP
     48 missense_variant|MODERATE|CAP1
     47 missense_variant|MODERATE|SYNE2
     47 missense_variant|MODERATE|DNAH14
     46 missense_variant|MODERATE|LGALS8
     45 missense_variant|MODERATE|TACC2
     45 missense_variant|MODERATE|MUC6


making problematic gene list

In [ ]:
!cat <<EOF > remove_genes.txt
HLA-
MUC
NBPF
PLEC
SYNE2
AHNAK2
DNAH
FBN
EOF

SyntaxError: invalid syntax (1135610558.py, line 2)

In [ ]:
%%bash
cat <<EOF > remove_genes.txt
HLA-
MUC
NBPF
PLEC
SYNE2
AHNAK2
DNAH
FBN
EOF

In [ ]:
!cat remove_genes.txt

HLA-
MUC
NBPF
PLEC
SYNE2
AHNAK2
DNAH
FBN


removing from moderate

In [ ]:
!grep "missense_variant" ann_final.tsv | grep "MODERATE" | grep -v -f remove_genes.txt > moderate_clean.tsv

checking results

In [ ]:
!cut -f3 moderate_clean.tsv | sort | uniq -c | sort -nr | head -20

    208 missense_variant|MODERATE|OBSCN
    167 missense_variant|MODERATE|TTN
    126 missense_variant|MODERATE|ZNF717
    101 missense_variant|MODERATE|NEB
     57 missense_variant|MODERATE|ASAH1
     54 missense_variant|MODERATE|PIGN
     54 missense_variant|MODERATE|FLG
     51 missense_variant|MODERATE|TSBP1
     48 missense_variant|MODERATE|FCGBP
     48 missense_variant|MODERATE|CAP1
     46 missense_variant|MODERATE|LGALS8
     45 missense_variant|MODERATE|TACC2
     43 missense_variant|MODERATE|SYNE1
     43 missense_variant|MODERATE|ADGRE1
     42 missense_variant|MODERATE|OR9G1
     42 missense_variant|MODERATE|FN1
     41 missense_variant|MODERATE|ATP7B
     38 missense_variant|MODERATE|PCDH15
     38 missense_variant|MODERATE|DMD
     37 missense_variant|MODERATE|ZNF568


we can also use following command to update gene list as filter is not completely applied above

In [ ]:
!echo -e "HLA-\nMUC\nNBPF\nPLEC\nSYNE\nAHNAK\nDNAH\nFBN\nTTN\nOBSCN\nNEB\nZNF\nDMD\nOR" > remove_genes.txt

cchecking results

In [ ]:
!grep "missense_variant" ann_final.tsv | grep "MODERATE" | grep -v -f remove_genes.txt > moderate_clean_v2.tsv

In [ ]:
!cut -f3 moderate_clean_v2.tsv | sort | uniq -c | sort -nr | head -20

     57 missense_variant|MODERATE|ASAH1
     54 missense_variant|MODERATE|PIGN
     54 missense_variant|MODERATE|FLG
     51 missense_variant|MODERATE|TSBP1
     48 missense_variant|MODERATE|FCGBP
     48 missense_variant|MODERATE|CAP1
     46 missense_variant|MODERATE|LGALS8
     45 missense_variant|MODERATE|TACC2
     43 missense_variant|MODERATE|ADGRE1
     42 missense_variant|MODERATE|FN1
     41 missense_variant|MODERATE|ATP7B
     38 missense_variant|MODERATE|PCDH15
     37 missense_variant|MODERATE|MKI67
     36 missense_variant|MODERATE|SIRPA
     36 missense_variant|MODERATE|PRUNE2
     36 missense_variant|MODERATE|CYP4A22
     35 missense_variant|MODERATE|SPATA22
     35 missense_variant|MODERATE|DST
     35 missense_variant|MODERATE|DMBT1
     34 missense_variant|MODERATE|MYLK


In [ ]:
!cut -f3 moderate_clean_v2.tsv | sort | uniq -c | sort -nr | head -20

     57 missense_variant|MODERATE|ASAH1
     54 missense_variant|MODERATE|PIGN
     54 missense_variant|MODERATE|FLG
     51 missense_variant|MODERATE|TSBP1
     48 missense_variant|MODERATE|FCGBP
     48 missense_variant|MODERATE|CAP1
     46 missense_variant|MODERATE|LGALS8
     45 missense_variant|MODERATE|TACC2
     43 missense_variant|MODERATE|ADGRE1
     42 missense_variant|MODERATE|FN1
     41 missense_variant|MODERATE|ATP7B
     38 missense_variant|MODERATE|PCDH15
     37 missense_variant|MODERATE|MKI67
     36 missense_variant|MODERATE|SIRPA
     36 missense_variant|MODERATE|PRUNE2
     36 missense_variant|MODERATE|CYP4A22
     35 missense_variant|MODERATE|SPATA22
     35 missense_variant|MODERATE|DST
     35 missense_variant|MODERATE|DMBT1
     34 missense_variant|MODERATE|MYLK


required for ML Analysis

In [ ]:
!bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%QUAL\t%DP\n' final_filtered_variants.vcf.gz > variants.tsv